# Building Machine Learning Models to compare to Baseline Bridge Equation

### Making all the neccessary imports of modules and data and setting it a fixed state for reproducibility.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)
torch.manual_seed(42)
np.random.seed(42)

### Once again get all the relevant titles / series. Remember that our Target variable is just the GDP official real change. Split, is for our model -- remember how we want to avoid look ahead bias (Notebook 2). 

In [3]:
SERIES = {
    "INDPRO": "log_diff", "RSAFS": "log_diff", "PAYEMS": "log_diff",
    "UNRATE": "diff", "HOUST": "log_diff", "PCE": "log_diff",
    "DGORDER": "log_diff", "ICSA": "log_diff", "T10Y2Y": "level",
}
TARGET = "A191RL1Q225SBEA"
SPLIT = "2015-01-01"

### Using Notebook 2 helper functions again, if you need justification refer to these functions in Notebook 2. 

In [4]:
def load_series(code):
    df = pd.read_csv(RAW / f"{code}.csv", index_col=0, parse_dates=True)
    s = df.iloc[:, 0]; s.name = code
    return s

def transform(s, rule):
    if rule == "log_diff":
        return np.log(s.replace(0, np.nan)).diff()
    if rule == "diff":
        return s.diff()
    return s

### Resampling every indcator to monthly and also transforming it because since we are trying to build DFM model, need high frequency data. 

In [5]:
def to_monthly(s):
    return s.resample("ME").mean()

def build_monthly(codes):
    cols = {c: transform(to_monthly(load_series(c)), rule)
            for c, rule in codes.items()}
    return pd.DataFrame(cols).dropna(how="all")

X_m = build_monthly(SERIES)

### Standardize our data because both DFM and LSTM Models we are thinking of buidling is prone to scaling in that LSTMS uses functions such as sigmoid and tanh which are sensitive to a range of like 0 to 1 million. DFM uses variance, why a large range then affects is is self - explanatory. 

In [6]:
def fit_scaler(df, cutoff):
    scaler = StandardScaler()
    scaler.fit(df.loc[df.index < cutoff].dropna())
    return scaler

def apply_scaler(df, scaler):
    arr = scaler.transform(df.fillna(df.mean()))
    return pd.DataFrame(arr, index=df.index, columns=df.columns)

scaler = fit_scaler(X_m, SPLIT)
X_m_scaled = apply_scaler(X_m, scaler)

### Fitting the Dynamic Factor Model so that way we can compute later on how well it does. 

In [8]:
def fit_dfm(X, cutoff):
    train = X.loc[X.index < cutoff].dropna()
    model = DynamicFactor(train, k_factors=1, factor_order=1)
    return model.fit(disp=False, maxiter=200)

dfm_res = fit_dfm(X_m_scaled, SPLIT)

### Running a Kalman Smoother over full dataset. The reaoson this is relevant is because a DFM assumes there is some hidden state that causes the observed data to be that way. Kalman algos essentially is used to estimate this hidden state (Learned in Stat 153 -- helps keep our DFM actually viable). Runs Kalman Filter (in cell above) than the Smoother. 

In [9]:
def smooth_factor(fitted, X: pd.DataFrame) -> pd.Series:
    applied = fitted.apply(X.fillna(X.mean()))
    factor = pd.Series(applied.factors.smoothed[0],
                       index=X.index, name="factor")
    return factor

factor_m = smooth_factor(dfm_res, X_m_scaled)

### Averaging each months values to quarters because DFM gives the monthly signals, but GDP only tracked quarterly. 

In [10]:
def to_quarterly(s):
    return s.resample("QE").mean()

def factor_panel(factor, target_code):
    f_q = to_quarterly(factor)
    y = load_series(target_code)
    y.index = y.index.to_period("Q").to_timestamp("Q")
    y.name = "gdp_growth"
    return pd.concat([f_q, y], axis=1).dropna()

panel_f = factor_panel(factor_m, TARGET)

### Now that signals are aligned, want to see how much the GDP growth moves when the factor is moved by an X amount. Essentially, DFM gave us the hidden signal (which we call THE factor) and now the "bridge" is translating that signal to an actual GDP Growth metric.

In [11]:
from sklearn.linear_model import LinearRegression

def fit_factor_bridge(panel: pd.DataFrame, cutoff: str):
    tr = panel.loc[panel.index < cutoff]
    X = tr[["factor"]].values; y = tr["gdp_growth"].values
    return LinearRegression().fit(X, y)

def predict_factor_bridge(model, panel: pd.DataFrame, cutoff: str):
    te = panel.loc[panel.index >= cutoff]
    pred = model.predict(te[["factor"]].values)
    return pred, te["gdp_growth"].values, te.index

f_bridge = fit_factor_bridge(panel_f, SPLIT)
dfm_pred, dfm_actual, dfm_idx = predict_factor_bridge(f_bridge, panel_f, SPLIT)

### Scoring the DFM Model to compare it to baseline bridge equation

In [12]:
def metrics(pred, actual):
    return {"rmse": float(np.sqrt(mean_squared_error(actual, pred))),
            "mae":  float(mean_absolute_error(actual, pred))}

dfm_metrics = metrics(dfm_pred, dfm_actual)
dfm_metrics

{'rmse': 4.327797227007992, 'mae': 2.205027642032334}

### Once again putting the monthly signals to quarterly (this time for LSTM) as this is what Hopp does where a minimal LSTM on the same quarterly inputs as the bridge equation isolates the effect of sequential modeling -- which we want as we want to compare LSTM scores directly to baseline.

In [13]:
def quarterly_panel(X_m, target_code):
    X_q = X_m.resample("QE").mean()
    y = load_series(target_code)
    y.index = y.index.to_period("Q").to_timestamp("Q")
    y.name = "gdp_growth"
    return pd.concat([X_q, y], axis=1).dropna()

panel_q = quarterly_panel(X_m_scaled, TARGET)

### LSTM uses sequences, so we need to make our data sequence windows. L = 4 is one year since its 4 quarters

In [14]:
L = 4
def make_sequences(df, length):
    feats = df.drop(columns=["gdp_growth"]).values
    tgt = df["gdp_growth"].values
    xs = [feats[i-length:i] for i in range(length, len(df))]
    ys = [tgt[i] for i in range(length, len(df))]
    idx = df.index[length:]
    return np.asarray(xs), np.asarray(ys), idx

### ENnsuring we do not mix up training data so any post 2015 is not in training set (once again to avoid look ahead bias)

In [15]:
def split_sequences(X, y, idx, cutoff):
    mask_tr = idx < pd.Timestamp(cutoff)
    return X[mask_tr], y[mask_tr], X[~mask_tr], y[~mask_tr], idx[~mask_tr]

X_all, y_all, idx_all = make_sequences(panel_q, L)
X_tr, y_tr, X_te, y_te, idx_te = split_sequences(X_all, y_all, idx_all, SPLIT)

### Building the LSTM model so we can actually use it?

In [16]:
class NowcastLSTM(nn.Module):
    def __init__(self, n_features: int, hidden: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=2,
                            batch_first=True, dropout=0.1)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

### Doing the Training stage, default is 300 epoch (how many runs) with a standard learning rate. Standard way of training the model. 

In [17]:
def train_lstm(model, X, y, epochs: int = 300, lr: float = 1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(Xt), yt)
        loss.backward(); opt.step()
    return model

n_features = X_tr.shape[2]
lstm = train_lstm(NowcastLSTM(n_features), X_tr, y_tr)

### Predictions of the LSTM, important to see since it shows if its actually good.

In [18]:
def predict_lstm(model, X):
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(X, dtype=torch.float32)).numpy()
    return out

lstm_pred = predict_lstm(lstm, X_te)
lstm_metrics = metrics(lstm_pred, y_te)
lstm_metrics

{'rmse': 7.098746182333617, 'mae': 3.4164959257976575}

### Saving the scores of the DFM and LSTM model so we do not have to rerun whole Notebook 3 each time for Notebook 4

In [19]:
def save_ml(dfm_pred, lstm_pred, y_te, idx_te, path):
    df = pd.DataFrame({"actual": y_te, "lstm_pred": lstm_pred}, index=idx_te)
    dfm_df = pd.DataFrame({"dfm_pred": dfm_pred}, index=dfm_idx)
    out = df.join(dfm_df, how="outer")
    out.to_csv(path)

save_ml(dfm_pred, lstm_pred, y_te, idx_te, OUT / "ml_predictions.csv")